In [1]:
import os
import dotenv
from pathlib import Path

from langchain_core.messages import AIMessage, HumanMessage
from langchain_community.document_loaders.text import TextLoader
from langchain_community.document_loaders import (
    WebBaseLoader, 
    PyPDFLoader, 
    Docx2txtLoader,
)
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

dotenv.load_dotenv()

USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [3]:
# Load docs

doc_paths = [
    "docs/deep_learning.pdf",
    "docs/deep_learning.docx",
    "docs/deep_learning.txt"
]

docs = [] 
for doc_file in doc_paths:
    file_path = Path(doc_file)

    try:
        if doc_file.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
        elif doc_file.endswith(".docx"):
            loader = Docx2txtLoader(file_path)
        elif doc_file.endswith(".txt") or doc_file.name.endswith(".md"):
            loader = TextLoader(file_path)
        else:
            print(f"Document type {doc_file.type} not supported.")
            continue

        docs.extend(loader.load())

    except Exception as e:
        print(f"Error loading document {doc_file.name}: {e}")


# Load URLs

url = "https://docs.streamlit.io/develop/quick-reference/release-notes"
try:
    loader = WebBaseLoader(url)
    docs.extend(loader.load())

except Exception as e:
    print(f"Error loading document from {url}: {e}")

In [4]:
docs

[Document(metadata={'source': 'docs\\deep_learning.pdf', 'page': 0}, page_content='Deep Learning  \nIn machine learning , deep learning  focuses on utilizing multilayered  neural networks  to perform \ntasks such as  classification , regression , and  representation learning . The field takes inspiration \nfrom  biological neuroscience  and revolves around stacking  artificial neurons  into layers and "training" \nthem to process data. The adjective "deep" refers to the use of multiple layers (ranging from three to \nseveral hundred or thousands) in the network. Methods used can be  supervised , semi -\nsupervised  or unsupervised .[2] \nSome common deep learning network architectures include  fully connected networks , deep belief \nnetworks , recurrent neural networks , convolutional neural networks , generative adversarial \nnetworks , transformers , and  neural radiance fields . These architectures have been applied to fields \nincluding  computer vision , speech recognition , natu

In [5]:
# Split docs

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=1000,
)

document_chunks = text_splitter.split_documents(docs)

In [6]:
# Tokenize and load the documents to the vector store

vector_db = Chroma.from_documents(
    documents=document_chunks,
    embedding=OpenAIEmbeddings(),
)

In [7]:
# Retrieve

def _get_context_retriever_chain(vector_db, llm):
    retriever = vector_db.as_retriever()
    prompt = ChatPromptTemplate.from_messages([
        MessagesPlaceholder(variable_name="messages"),
        ("user", "{input}"),
        ("user", "Given the above conversation, generate a search query to look up in order to get inforamtion relevant to the conversation, focusing on the most recent messages."),
    ])
    retriever_chain = create_history_aware_retriever(llm, retriever, prompt)

    return retriever_chain

In [8]:
def get_conversational_rag_chain(llm):
    retriever_chain = _get_context_retriever_chain(vector_db, llm)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
        """You are a helpful assistant. You will have to answer to user's queries.
        You will have some context to help with your answers, but now always would be completely related or helpful.
        You can also use your knowledge to assist answering the user's queries.\n
        {context}"""),
        MessagesPlaceholder(variable_name="messages"),
        ("user", "{input}"),
    ])
    stuff_documents_chain = create_stuff_documents_chain(llm, prompt)

    return create_retrieval_chain(retriever_chain, stuff_documents_chain)

In [9]:
# Augmented Generation

llm_stream_openai = ChatOpenAI(
    model="gpt-4o-mini",  # Here you could use "o1-preview" or "o1-mini" if you already have access to them
    temperature=0.3,
    streaming=True,
)

llm_stream_anthropic = ChatAnthropic(
    model="claude-3-5-sonnet-20240620",
    temperature=0.3,
    streaming=True,
)

llm_stream = llm_stream_openai  # Select between OpenAI and Anthropic models for the response

messages = [
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Hi there! How can I assist you today?"},
    {"role": "user", "content": "Tell me about finding in deep learning in the year of 2000s."},
]
messages = [HumanMessage(content=m["content"]) if m["role"] == "user" else AIMessage(content=m["content"]) for m in messages]

conversation_rag_chain = get_conversational_rag_chain(llm_stream)
response_message = "*(RAG Response)*\n"
for chunk in conversation_rag_chain.pick("answer").stream({"messages": messages[:-1], "input": messages[-1].content}):
    response_message += chunk
    print(chunk, end="", flush=True)

messages.append({"role": "assistant", "content": response_message})

In the 2000s, deep learning experienced a period of relative stagnation compared to the rapid advancements seen in the 1990s. During this time, simpler models, such as support vector machines (SVMs) and task-specific handcrafted features, became more popular due to the computational costs associated with neural networks and a lack of understanding of how to effectively train deep architectures.

However, there were still significant developments in deep learning during this decade:

1. **LSTM Networks**: Long Short-Term Memory (LSTM) networks, which were introduced in the mid-1990s, began to show competitive performance with traditional speech recognizers on certain tasks by the early 2000s. In 2006, researchers combined LSTMs with connectionist temporal classification (CTC), which improved their performance further.

2. **Deep Belief Networks (DBNs)**: In 2006, Geoffrey Hinton and his colleagues developed deep belief networks, which were trained using a layer-by-layer approach with re